# Lekcja 2.3 — eksperymenty: BFS i UCS w zatłoczonym magazynie

Uruchamiaj komórki po kolei. Przed każdym eksperymentem zapisz swoje przewidywanie w komórce „Twoje przewidywanie” (kliknij ją dwukrotnie, żeby edytować), a dopiero potem uruchom kod.

Pierwsza komórka wybiera implementację:

- `"mine"` używa Twojego `task.py`; najpierw rozwiąż część B zadania;
- `"reference"` używa implementacji wzorcowej kursu. Wpisz `IMPLEMENTATION = "reference"`, jeśli chcesz uruchomić notebook przed skończeniem zadania.

In [ ]:
%load_ext autoreload
%autoreload 2
import os

# "mine" uses your task.py; "reference" uses the course's reference implementation.
IMPLEMENTATION = os.environ.get("INTROAI_IMPLEMENTATION", "mine")
if IMPLEMENTATION == "mine":
    import task as algorithms
else:
    from introai.search import reference as algorithms

Funkcja `run` uruchamia jedno przeszukiwanie na mapie i zwraca węzeł celu oraz listę rozwiniętych węzłów. Funkcja `summary` wypisuje liczbę kroków trasy, jej koszt i liczbę rozwinięć. BFS pochodzi zawsze z implementacji wzorcowej kursu, a UCS z implementacji wybranej w pierwszej komórce. Na rysunkach niebieskie pola to pola rozwinięte, pomarańczowe to trasa, a zielone, kreskowane pola to strefy wolnego ruchu.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt

from introai.search.reference import bfs
from introai.trace import TraceLimitExceeded, record
from introai.world.plot import plot_map
from introai.world.warehouse import RouteProblem, WarehouseMap, render_text

MAPS = Path.cwd().parents[2] / "data" / "maps"


def run(algorithm, warehouse, limit=100_000):
    """Run one search on a map; return the goal node and the expanded nodes."""
    try:
        with record(limit=limit, kinds={"expand"}) as trace:
            node = algorithm(RouteProblem(warehouse))
    except TraceLimitExceeded:
        raise RuntimeError(
            f"{algorithm.__name__} expanded more than {limit} nodes - check task.py "
            '(or set IMPLEMENTATION = "reference" in the first cell)'
        ) from None
    except TypeError as error:
        if "not supported between instances" not in str(error):
            raise
        raise RuntimeError(
            f"{algorithm.__name__} compared two Node objects - put a counter between "
            "the priority and the node on the heap "
            '(or set IMPLEMENTATION = "reference" in the first cell)'
        ) from None
    if node is None:
        raise RuntimeError(
            f"{algorithm.__name__} found no route, but every map here has one - check task.py "
            '(or set IMPLEMENTATION = "reference" in the first cell)'
        )
    return node, [event.data["node"] for event in trace.of_kind("expand")]


def summary(label, node, expanded):
    print(
        f"{label}: {len(node.solution())} steps, cost {node.path_cost:g}, expanded {len(expanded)}"
    )

## Eksperyment 1: korytarz czy objazd

Robot `R` ma dojechać do rampy `D`. Prosto prowadzi korytarz pełen stref wolnego ruchu `~`, a dookoła biegną dwie zwykłe alejki.

In [ ]:
congested = WarehouseMap.from_file(MAPS / "congested.txt")
print(render_text(congested))

**Twoje przewidywanie.** Którędy pojedzie robot według BFS, a którędy według UCS? Ile będzie kosztować każda trasa? Który algorytm rozwinie więcej pól?

*Moje przewidywanie:* …

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(10, 6))
for ax, (label, algorithm) in zip(axes, [("BFS", bfs), ("UCS", algorithms.ucs)]):
    node, expanded = run(algorithm, congested)
    summary(label, node, expanded)
    path = node.states()
    plot_map(congested, path=path, expanded=[n.state for n in expanded], title=label, ax=ax)
fig.tight_layout()

**Do przemyślenia.** Porównaj wynik ze swoim przewidywaniem. Dlaczego BFS wybrał korytarz, choć był dużo droższy? Przez ile pól strefy wolnego ruchu przejeżdża trasa UCS i dlaczego nie da się ich uniknąć?

## Eksperyment 2: jak krótka strefa wolnego ruchu?

Funkcja `corridor_map(slow)` buduje tę samą mapę, ale strefa wolnego ruchu zajmuje w korytarzu tylko `slow` pól, tuż przed rampą. Reszta korytarza to zwykła podłoga.

**Twoje przewidywanie.** Przy jakiej długości strefy wolnego ruchu UCS przestanie jechać korytarzem i wybierze objazd? Spróbuj to policzyć, zanim uruchomisz kod.

*Moje przewidywanie:* …

In [ ]:
def corridor_map(slow):
    """congested.txt with only the last `slow` corridor cells, next to the dock, slow."""
    lines = (MAPS / "congested.txt").read_text().splitlines()
    row = lines[3]
    start, end = row.index("R") + 1, row.index("D")
    lines[3] = row[:start] + "." * (end - start - slow) + "~" * slow + row[end:]
    return WarehouseMap.from_text("\n".join(lines))


slow_cells = list(range(18))
bfs_costs, ucs_costs = [], []
print(f"{'slow':>4} {'BFS steps':>9} {'BFS cost':>8} {'UCS steps':>9} {'UCS cost':>8}")
for slow in slow_cells:
    warehouse = corridor_map(slow)
    bfs_node, _ = run(bfs, warehouse)
    ucs_node, _ = run(algorithms.ucs, warehouse)
    bfs_costs.append(bfs_node.path_cost)
    ucs_costs.append(ucs_node.path_cost)
    print(
        f"{slow:>4} {len(bfs_node.solution()):>9} {bfs_node.path_cost:>8g} "
        f"{len(ucs_node.solution()):>9} {ucs_node.path_cost:>8g}"
    )

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
for label, costs, color in [("BFS", bfs_costs, "#2a78d6"), ("UCS", ucs_costs, "#eb6834")]:
    ax.plot(
        slow_cells,
        costs,
        color=color,
        linewidth=2,
        marker="o",
        markersize=8,
        markeredgecolor="#fcfcfb",
        markeredgewidth=1.5,
        label=label,
    )
    ax.annotate(
        label,
        (slow_cells[-1], costs[-1]),
        textcoords="offset points",
        xytext=(8, 0),
        va="center",
        color="#0b0b0b",
    )
ax.set_title("Cost of the route found, by length of the slow zone", loc="left", color="#0b0b0b")
ax.set_xlabel("slow cells in the corridor", color="#52514e")
ax.set_xticks(range(0, 18, 2))
ax.set_ylabel("route cost", color="#52514e")
ax.grid(axis="y", color="#e1e0d9", linewidth=0.8)
ax.set_axisbelow(True)
for side in ("top", "right"):
    ax.spines[side].set_visible(False)
for side in ("left", "bottom"):
    ax.spines[side].set_color("#c3c2b7")
ax.tick_params(colors="#52514e")
ax.margins(x=0.08)
ax.legend(frameon=False, loc="upper left", labelcolor="#0b0b0b")
fig.tight_layout()

**Do przemyślenia.** Gdzie linie się rozchodzą i dlaczego właśnie tam? Ile kosztuje objazd i dlaczego jego koszt rośnie dopiero przy strefie z 17 pól? Czy przy jakiejś długości strefy obie trasy kosztują tyle samo? Którą wtedy wybiera UCS i od czego to zależy?

## Eksperyment 3: dokąd zagląda UCS

Na rysunku z eksperymentu 1 UCS rozwinął też pola w magazynie po lewej stronie, choć rampa jest po prawej.

**Twoje przewidywanie.** Które pola rozwija UCS, zanim zdejmie cel? Czy da się to opisać jednym warunkiem na koszt?

*Moje przewidywanie:* …

In [ ]:
node, expanded = run(algorithms.ucs, congested)
free = [
    (row, col)
    for row in range(congested.rows)
    for col in range(congested.cols)
    if congested.is_free((row, col))
]
print(f"free cells: {len(free)}, expanded by UCS: {len(expanded)}")
print(f"cost of the route: {node.path_cost:g}")
print(f"the most expensive expanded node: {max(n.path_cost for n in expanded):g}")
storage = [n for n in expanded if n.state[1] < congested.start[1]]
print(f"expanded cells in the storage area left of R: {len(storage)}")

**Do przemyślenia.** Jaki jest najwyższy koszt rozwiniętego węzła w porównaniu z kosztem trasy? Dlaczego UCS musi zajrzeć do każdego pola, do którego da się dojechać taniej niż do celu, nawet jeśli leży w przeciwnym kierunku? Jaka informacja pozwoliłaby mu tego uniknąć? Odpowiedź pojawi się w lekcji 2.4.

## Podsumowanie

Wróć do pytań kontrolnych w lekcji. Po własnej próbie możesz porównać swoje wyniki i wnioski z odpowiedziami w `solutions/02_search/03_uniform_cost_search/answers.pl.md`.